In [2]:
import os
import rasterio
import numpy as np

In [3]:
data_root = "/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/Sen1Floods1_data/v1.1/data/flood_events/HandLabeled/"

# Check one chip
s1_path  = os.path.join(data_root, "S1Hand/Bolivia_103757_S1Hand.tif")
lbl_path = os.path.join(data_root, "LabelHand/Bolivia_103757_LabelHand.tif")

with rasterio.open(s1_path) as src:
    s1 = src.read()           # shape: (2, 512, 512) → VV, VH
    print("S1 shape  :", s1.shape)
    print("S1 dtype  :", src.dtypes)
    print("S1 CRS    :", src.crs)
    print("S1 res    :", src.res)

with rasterio.open(lbl_path) as src:
    label = src.read(1)       # shape: (512, 512)
    unique = np.unique(label)
    print("Label values:", unique)   # Expected: [-1, 0, 1]

S1 shape  : (2, 512, 512)
S1 dtype  : ('float32', 'float32')
S1 CRS    : EPSG:4326
S1 res    : (8.983152841195215e-05, 8.983152841195215e-05)
Label values: [-1  0  1]


In [4]:
import pandas as pd
import glob

# Load official splits
# splits_df = pd.read_csv("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/Sen1Floods11_data/v1.1/splits/flood_handlabeled/flood_bolivia_data.csv")
# print(splits_df.groupby(["split", "country"]).size().unstack(fill_value=0))

# Count downloaded chips
s1_chips = glob.glob("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/data/flood_events/HandLabeled/S1Hand/*.tif", recursive=True)
print(f"\nTotal S1 chips downloaded: {len(s1_chips)}")
# Expected: 446 for HandLabeled


Total S1 chips downloaded: 446


In [ ]:
# Rename label tiles
from pathlib import Path
LABEL_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Labels")

# Use a specific glob pattern to only target files ending with AgDamage
for file in LABEL_DIR.rglob("*_LabelAgDamage.tif"):
    chip_id = file.stem.replace("_LabelAgDamage", "")
    new_name = f"LabelMask_{chip_id}.tif"
    
    new_path = file.with_name(new_name)
    
    file.rename(new_path)
    print(f"Renamed: {file.name} -> {new_name}")


Renamed: USA_11422_LabelAgDamage.tif -> LabelMask_USA_11422.tif
Renamed: USA_58086_LabelAgDamage.tif -> LabelMask_USA_58086.tif
Renamed: USA_86502_LabelAgDamage.tif -> LabelMask_USA_86502.tif
Renamed: USA_170264_LabelAgDamage.tif -> LabelMask_USA_170264.tif
Renamed: USA_217598_LabelAgDamage.tif -> LabelMask_USA_217598.tif
Renamed: USA_224165_LabelAgDamage.tif -> LabelMask_USA_224165.tif
Renamed: USA_225017_LabelAgDamage.tif -> LabelMask_USA_225017.tif
Renamed: USA_348639_LabelAgDamage.tif -> LabelMask_USA_348639.tif
Renamed: USA_387945_LabelAgDamage.tif -> LabelMask_USA_387945.tif
Renamed: USA_430764_LabelAgDamage.tif -> LabelMask_USA_430764.tif
Renamed: USA_595451_LabelAgDamage.tif -> LabelMask_USA_595451.tif
Renamed: USA_638521_LabelAgDamage.tif -> LabelMask_USA_638521.tif
Renamed: USA_646878_LabelAgDamage.tif -> LabelMask_USA_646878.tif
Renamed: USA_652955_LabelAgDamage.tif -> LabelMask_USA_652955.tif
Renamed: USA_741178_LabelAgDamage.tif -> LabelMask_USA_741178.tif
Renamed: USA_761

In [ ]:
# Rename Before tiles provided by Team
from pathlib import Path
OLD_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Before/S1GRD/")
for file in OLD_DIR.rglob("*_V2.tif"):
    chip_id = file.stem.replace("_V2", "")
    new_name = f"{chip_id}.tif"
    new_path = file.with_name(new_name)
    
    file.rename(new_path)
    print(f"Renamed: {file.name} -> {new_name}")

In [ ]:
#Copy After tiles from original location to project input folder to train
import shutil
from pathlib import Path

# Paths already defined in your script
LABEL_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Labels")
AFTER_S1 = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/data/flood_events/HandLabeled/S1Hand/")
AFTER_S2 = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/data/flood_events/HandLabeled/S2Hand/")

S1_NEW = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/After/S1GRD")
S2_NEW = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/After/S2L2A")

# Ensure destination directories exist
S1_NEW.mkdir(parents=True, exist_ok=True)
S2_NEW.mkdir(parents=True, exist_ok=True)

for file in LABEL_DIR.rglob("LabelMask_*.tif"):
    chip_id = file.stem.replace("LabelMask_", "")
    
    # Construct filenames
    s1_filename = f"{chip_id}_S1Hand.tif"
    s2_filename = f"{chip_id}_S2Hand.tif" # Fixed from S1 to S2

    # Define Source and Destination full paths using the / operator
    src_s1 = AFTER_S1 / s1_filename
    dst_s1 = S1_NEW / s1_filename

    src_s2 = AFTER_S2 / s2_filename
    dst_s2 = S2_NEW / s2_filename

    # Copy files if they exist in the source directory
    if src_s1.exists():
        shutil.copy2(src_s1, dst_s1)
        print(f"Copied S1: {s1_filename}")
    else:
        print(f"Warning: {src_s1} not found")

    if src_s2.exists():
        shutil.copy2(src_s2, dst_s2)
        print(f"Copied S2: {s2_filename}")
    else:
        print(f"Warning: {src_s2} not found")


Copied S1: USA_11422_S1Hand.tif
Copied S2: USA_11422_S2Hand.tif
Copied S1: USA_58086_S1Hand.tif
Copied S2: USA_58086_S2Hand.tif
Copied S1: USA_86502_S1Hand.tif
Copied S2: USA_86502_S2Hand.tif
Copied S1: USA_170264_S1Hand.tif
Copied S2: USA_170264_S2Hand.tif
Copied S1: USA_217598_S1Hand.tif
Copied S2: USA_217598_S2Hand.tif
Copied S1: USA_224165_S1Hand.tif
Copied S2: USA_224165_S2Hand.tif
Copied S1: USA_225017_S1Hand.tif
Copied S2: USA_225017_S2Hand.tif
Copied S1: USA_348639_S1Hand.tif
Copied S2: USA_348639_S2Hand.tif
Copied S1: USA_387945_S1Hand.tif
Copied S2: USA_387945_S2Hand.tif
Copied S1: USA_430764_S1Hand.tif
Copied S2: USA_430764_S2Hand.tif
Copied S1: USA_595451_S1Hand.tif
Copied S2: USA_595451_S2Hand.tif
Copied S1: USA_638521_S1Hand.tif
Copied S2: USA_638521_S2Hand.tif
Copied S1: USA_646878_S1Hand.tif
Copied S2: USA_646878_S2Hand.tif
Copied S1: USA_652955_S1Hand.tif
Copied S2: USA_652955_S2Hand.tif
Copied S1: USA_741178_S1Hand.tif
Copied S2: USA_741178_S2Hand.tif
Copied S1: USA_7

In [58]:
from pathlib import Path

# Define the 5 locations
DIRS = {
    "Label":  Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Labels"),
    "S1_After":  Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/After/S1GRD"),
    "S2_After":  Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/After/S2L2A"),
    "S1_Before": Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Before/S1GRD"),
    "S2_Before": Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Before/S2L2A")
}

missing_count = 0

print("--- Starting Dataset Verification ---")

# We use the Labels directory as the "Source of Truth"
for label_file in DIRS["Label"].glob("LabelMask_*.tif"):
    chip_id = label_file.stem.replace("LabelMask_", "")
    
    # Define expected filenames for this chip_id in other folders
    expected_files = {
        "S1_After":  f"S1After_{chip_id}.tif",
        "S2_After":  f"S2After_{chip_id}.tif",
        "S1_Before": f"S1Before_{chip_id}.tif", # Adjust suffix if 'Before' uses different naming
        "S2_Before": f"S2Before_{chip_id}.tif"
    }

    # Check each location
    for key, filename in expected_files.items():
        path_to_check = DIRS[key] / filename
        if not path_to_check.exists():
            print(f"[MISSING] Chip: {chip_id} | Location: {key} | Expected: {filename}")
            missing_count += 1

if missing_count == 0:
    print("Verification Complete: All chips have matching pairs in all 5 locations!")
else:
    print(f"Verification Complete: Found {missing_count} missing files.")


--- Starting Dataset Verification ---
Verification Complete: All chips have matching pairs in all 5 locations!
